# Atividade 01

In [1]:
import matplotlib.pyplot as plt  # noqa: F401
import numpy as np  # noqa: F401
import pandas as pd  # noqa: F401
import polars as pl  # noqa: F401
import seaborn as sns
import sklearn as sk  # noqa: F401

sns.set_theme()

## ENTRADA DE DADOS

In [6]:
ENDERECO_DADOS = 'https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv'

# LER O ARQUIVO CSV COM POLARS
df_ocorrencias = pl.read_csv(
    ENDERECO_DADOS,
    separator=';', 
    encoding='iso-8859-1'
    )

# SELECIONAR AS COLUNAS DE INTERESSE
df_veiculos = df_ocorrencias[['cisp', 'roubo_veiculo', 'recuperacao_veiculos']]

# Agrupar e totalizar os roubos e recuperações por CISP (Batalhao) no DF NOVO  
df_total_veiculos = df_veiculos.group_by('cisp').agg(
    pl.col('roubo_veiculo').sum(),
    pl.col('recuperacao_veiculos').sum()
)
print(df_total_veiculos)

shape: (138, 3)
┌──────┬───────────────┬──────────────────────┐
│ cisp ┆ roubo_veiculo ┆ recuperacao_veiculos │
│ ---  ┆ ---           ┆ ---                  │
│ i64  ┆ i64           ┆ i64                  │
╞══════╪═══════════════╪══════════════════════╡
│ 134  ┆ 1948          ┆ 3136                 │
│ 137  ┆ 6             ┆ 90                   │
│ 12   ┆ 323           ┆ 872                  │
│ 9    ┆ 4701          ┆ 1952                 │
│ 15   ┆ 1268          ┆ 1488                 │
│ …    ┆ …             ┆ …                    │
│ 101  ┆ 47            ┆ 124                  │
│ 119  ┆ 791           ┆ 1023                 │
│ 128  ┆ 1964          ┆ 1966                 │
│ 122  ┆ 185           ┆ 283                  │
│ 125  ┆ 942           ┆ 1305                 │
└──────┴───────────────┴──────────────────────┘


## PROCESSAMENTO DOS DADOS

### Tratamento dos dados

Excluir uma parte de valores extremos (OUTLIER SUPERIORES) do DF_roubo e DF_recuperacao para não distorcer a análise de correlação

In [10]:
# Calculando outliers
# Limite superior pelo método IQR (Q3 + 1,5 × IQR)


# >>> ROUBO DE VEÍCULOS
q1_roubo = df_total_veiculos['roubo_veiculo'].quantile(0.25)
q3_roubo = df_total_veiculos['roubo_veiculo'].quantile(0.75)
limite_roubo = q3_roubo + 1.5 * (q3_roubo - q1_roubo)

# >>> RECUPERAÇÃO DE VEÍCULOS
q1_recuperacao = df_total_veiculos['recuperacao_veiculos'].quantile(0.25)
q3_recuperacao = df_total_veiculos['recuperacao_veiculos'].quantile(0.75)
limite_recuperacao = q3_recuperacao + 1.5 * (q3_recuperacao - q1_recuperacao)



# Remover linhas que sejam outliers em qualquer uma das colunas
df_total_veiculos_sem_outliers = df_total_veiculos.filter(
    (pl.col('roubo_veiculo') <= limite_roubo)
    & (pl.col('recuperacao_veiculos') <= limite_recuperacao)
)

# Transformando em Arrays
array_roubo_veiculo = np.array(df_total_veiculos_sem_outliers['roubo_veiculo'])
array_recuperacao_veiculos = np.array(df_total_veiculos_sem_outliers['recuperacao_veiculos'])

# Transformando em Arrays com outliers
array_roubo_veiculo_outliers = np.array(df_total_veiculos['roubo_veiculo'])
array_recuperacao_veiculos_outliers = np.array(df_total_veiculos['recuperacao_veiculos'])

### 

In [11]:
# Calculando a correlação sem outliers 
correlacao = np.corrcoef(array_roubo_veiculo, array_recuperacao_veiculos)[0, 1]
print('Correlação: ', correlacao)

# Calculando a correlação com outliers
correlacao_outliers = np.corrcoef(array_roubo_veiculo_outliers, array_recuperacao_veiculos_outliers)[0, 1]
print('Correlação com outliers: ', correlacao_outliers)

Correlação:  0.9228772779263678
Correlação com outliers:  0.9271294220058007


## Criando o Modelo MACHINE LEARNING

In [ ]:
# Dividir em treino e teste

# Roubo de veículos (X): Variável independente - Utilizo para prever
# Recuperação de veículos (Y): Variável dependente - a queremos prever

X_train, X_test, y_train, y_test =(
    sk.train_test_split(
        array_roubo_veiculo,       # Variável independente
        array_recuperacao_veiculos,# Variável dependente  
        test_size=0.2,             # tamanho do conjunto de teste, logo o treino terá 0.8
        random_state=42
    )
)